# Split into a site workbook

## In short

This notebook puts the all-sites master CSV into one Excel workbook. The workbook has one sheet with all the data (`Combined`) and one sheet for each site. Use it to share the data with people who work in Excel.

To use it, set the input file and the output file at the top of the code cell. Then run the cell.

## Details

**Input**

The all-sites master CSV from `2_combine_sites.ipynb`. The CSV must have a `Site` column and a `Date` column.

**Output**

An Excel workbook (`.xlsx`) with these sheets:

| Sheet | Contents |
|---|---|
| `Combined` | All rows |
| One sheet for each site | The rows for that site |

**Packages**

This notebook also needs `xlsxwriter` and `tqdm`.

**How to run**

1. Set `input_file` to the all-sites master CSV.
2. Set `output_file` to the path of the workbook to create.
3. To make sheets only for some sites, set `sites` (for example `["RoujolA", "RoujolD"]`). This step is optional.
4. Run the cell.

**Notes**

- The `Date` column is written as an Excel date with the format `mm/dd/yy`. If the notebook cannot read a date, the cell is empty.
- The notebook removes the columns in `drop_columns` (`Date_original` and similar) if they exist.
- Empty values become empty cells.
- An Excel sheet name can have a maximum of 31 characters.
- A large dataset can take some minutes. A progress bar shows the progress for each sheet.


In [ ]:
import pandas as pd
from pathlib import Path
import xlsxwriter
from tqdm.auto import tqdm
import math

# -----------------------------
# Input and output file paths
# -----------------------------
# All-sites master CSV from 2_combine_sites.
input_file = Path('path/to/MSorRGB_MasterDataset_AllSites.csv')

# Excel workbook to create.
output_file = Path('path/to/MSorRGB_MasterDataset_Split_By_Site.xlsx')

# -----------------------------
# Settings
# -----------------------------
site_column = "Site"
date_column = "Date"

# One sheet is made for each site in this list, for example ["RoujolA", "RoujolD"].
# None makes one sheet for each site in the Site column.
sites = None

# Columns to remove if they exist
drop_columns = [
    "Date_original",
    "Original Date",
    "original date",
    "Original_Date"
]

# -----------------------------
# Read CSV
# -----------------------------
print("Reading CSV...")

df = pd.read_csv(
    input_file,
    low_memory=False
)

print(f"Loaded {len(df):,} rows and {len(df.columns):,} columns.")

# -----------------------------
# Remove unnecessary original-date columns
# -----------------------------
existing_drop_columns = [
    col for col in drop_columns
    if col in df.columns
]

if existing_drop_columns:
    print(f"Removing unnecessary column(s): {existing_drop_columns}")
    df = df.drop(columns=existing_drop_columns)

# -----------------------------
# Validate required columns
# -----------------------------
if site_column not in df.columns:
    print("Available columns:")
    print(df.columns.tolist())
    raise ValueError(f"Column '{site_column}' was not found in the dataset.")

if date_column not in df.columns:
    print("Available columns:")
    print(df.columns.tolist())
    raise ValueError(f"Column '{date_column}' was not found in the dataset.")

# Convert Date column to datetime only for Excel date formatting
df[date_column] = pd.to_datetime(
    df[date_column],
    errors="coerce"
)

print("\nRows per site:")
print(df[site_column].value_counts(dropna=False))

# -----------------------------
# Helper: write dataframe
# -----------------------------
def write_dataframe_to_sheet(workbook, sheet_name, dataframe):
    worksheet = workbook.add_worksheet(sheet_name)

    # Formats
    date_format = workbook.add_format({
        "num_format": "mm/dd/yy"
    })

    # Write headers
    for col_idx, col_name in enumerate(dataframe.columns):
        worksheet.write(0, col_idx, col_name)

    date_col_idx = dataframe.columns.get_loc(date_column)

    # Write data rows with progress bar
    for row_idx, row in tqdm(
        enumerate(dataframe.itertuples(index=False, name=None), start=1),
        total=len(dataframe),
        desc=f"Writing {sheet_name}",
        unit="row"
    ):
        for col_idx, value in enumerate(row):

            # Blank cells
            if pd.isna(value):
                worksheet.write_blank(row_idx, col_idx, None)

            # Date column: write as Excel date with mm/dd/yy format
            elif col_idx == date_col_idx:
                worksheet.write_datetime(
                    row_idx,
                    col_idx,
                    value.to_pydatetime(),
                    date_format
                )

            # Numeric values stay numeric
            elif isinstance(value, (int, float)) and not isinstance(value, bool):
                if isinstance(value, float) and math.isnan(value):
                    worksheet.write_blank(row_idx, col_idx, None)
                else:
                    worksheet.write_number(row_idx, col_idx, value)

            # Everything else stays as-is
            else:
                worksheet.write(row_idx, col_idx, value)

# -----------------------------
# Create workbook
# -----------------------------
print("\nCreating workbook...")

workbook = xlsxwriter.Workbook(
    output_file,
    {
        "strings_to_numbers": False,
        "strings_to_dates": False,
        "strings_to_formulas": False,
        "strings_to_urls": False,
        "constant_memory": True,
    }
)

# Combined/original sheet
write_dataframe_to_sheet(workbook, "Combined", df)

# Site-specific sheets
if sites is None:
    sites = sorted(df[site_column].dropna().astype(str).unique())

for site in sites:
    site_df = df[df[site_column] == site]
    print(f"\n{site}: {len(site_df):,} rows")
    write_dataframe_to_sheet(workbook, site, site_df)

# -----------------------------
# Save workbook
# -----------------------------
print("\nSaving workbook...")
workbook.close()

print("\nWorkbook created successfully:")
print(output_file)